# SaludAI - Getting Started

Este notebook demuestra los componentes fundamentales de SaludAI:

1. **FHIR Client** — Conectar a un servidor FHIR, buscar y leer recursos
2. **Terminology Resolver** — Resolver terminos clinicos a codigos SNOMED CT, CIE-10 y LOINC
3. **Query Builder** — Construir queries FHIR complejas con una API fluent

## Prerequisitos

```bash
# Levantar HAPI FHIR con datos argentinos sinteticos
docker compose up -d

# Esperar ~30 segundos a que carguen los datos
```

## 1. FHIR Client

`FHIRClient` es el cliente async para interactuar con cualquier servidor FHIR R4.

In [ ]:
from saludai_core.config import FHIRConfig
from saludai_core.fhir_client import FHIRClient

# Configurar conexion al servidor FHIR local
config = FHIRConfig(fhir_server_url="http://localhost:8080/fhir")

# Verificar conexion
async with FHIRClient(config=config) as client:
    capability = await client.check_connection()
    print(f"Servidor FHIR conectado")
    print(f"  Version: {capability.fhirVersion}")
    print(f"  Software: {capability.software.name if capability.software else 'N/A'}")

### Buscar pacientes

`search()` retorna un Bundle FHIR como diccionario. Podemos buscar por nombre, estado, o cualquier parametro FHIR.

In [ ]:
async with FHIRClient(config=config) as client:
    # Buscar primeros 5 pacientes
    bundle = await client.search("Patient", {"_count": "5"})

    print(f"Total de pacientes en el sistema: {bundle.get('total', 'N/A')}")
    print(f"Mostrando: {len(bundle.get('entry', []))} pacientes\n")

    for entry in bundle.get("entry", []):
        patient = entry["resource"]
        name = patient.get("name", [{}])[0]
        given = " ".join(name.get("given", []))
        family = name.get("family", "")
        birth = patient.get("birthDate", "N/A")
        gender = patient.get("gender", "N/A")
        print(f"  {given} {family} | Nacimiento: {birth} | Genero: {gender}")

### Buscar condiciones clinicas

Podemos buscar condiciones por codigo SNOMED CT y seguir las referencias a pacientes.

In [ ]:
async with FHIRClient(config=config) as client:
    # Buscar condiciones de diabetes tipo 2 (SNOMED CT: 44054006)
    bundle = await client.search("Condition", {
        "code": "http://snomed.info/sct|44054006",
        "_count": "10",
        "_include": "Condition:subject",  # Incluir pacientes referenciados
    })

    conditions = []
    patients = {}

    for entry in bundle.get("entry", []):
        resource = entry["resource"]
        if resource["resourceType"] == "Patient":
            patients[f"Patient/{resource['id']}"] = resource
        else:
            conditions.append(resource)

    print(f"Condiciones de diabetes tipo 2 encontradas: {len(conditions)}")
    print(f"Pacientes incluidos via _include: {len(patients)}\n")

    for cond in conditions[:5]:
        subject_ref = cond.get("subject", {}).get("reference", "")
        patient = patients.get(subject_ref, {})
        name = patient.get("name", [{}])[0]
        patient_name = " ".join(name.get("given", [])) + " " + name.get("family", "")
        onset = cond.get("onsetDateTime", "N/A")[:10] if cond.get("onsetDateTime") else "N/A"
        print(f"  Paciente: {patient_name.strip()} | Inicio: {onset}")

### Leer un recurso individual

`read()` obtiene un recurso FHIR por su ID y lo retorna como modelo `fhir.resources`.

In [ ]:
async with FHIRClient(config=config) as client:
    # Primero, obtener un ID de paciente de la busqueda
    bundle = await client.search("Patient", {"_count": "1"})
    patient_id = bundle["entry"][0]["resource"]["id"]

    # Leer el recurso completo (retorna modelo fhir.resources)
    patient = await client.read("Patient", patient_id)
    print(f"Tipo: {patient.get_resource_type()}")
    print(f"ID: {patient.id}")
    print(f"Nombre: {patient.name[0].given[0]} {patient.name[0].family}")
    print(f"Fecha de nacimiento: {patient.birthDate}")
    print(f"Genero: {patient.gender}")

    if patient.address:
        addr = patient.address[0]
        print(f"Direccion: {addr.city}, {addr.state}, {addr.country}")

## 2. Terminology Resolver

El `TerminologyResolver` mapea terminos clinicos en espanol (e ingles) a codigos estandar: SNOMED CT (edicion argentina), CIE-10 y LOINC.

In [ ]:
from saludai_core.terminology import TerminologyResolver

resolver = TerminologyResolver()
print(f"Sistemas cargados: {[s.value for s in resolver.get_systems()]}")
print(f"Total de conceptos: {resolver.concept_count}")

### Resolver terminos clinicos

`resolve()` encuentra la mejor coincidencia para un termino. Soporta nombres exactos, alias coloquiales y fuzzy matching.

In [ ]:
# Probar con distintos terminos clinicos
terms = [
    "diabetes tipo 2",
    "presion alta",
    "Diabetes mellitus tipo 2",  # Nombre formal
    "asma",
    "glucosa en sangre",         # Termino LOINC
    "E11",                       # Codigo CIE-10 directo
]

for term in terms:
    match = resolver.resolve(term)
    if match.concept:
        print(f"  '{term}'")
        display = match.concept.display
        system = match.concept.system.value
        print(f"    -> {display} ({system})")
        code = match.concept.code
        mtype = match.match_type.value
        print(f"    -> Codigo: {code} | Score: {match.score:.0f} | Tipo: {mtype}")
        print()

### Buscar multiples resultados

`search()` retorna las mejores coincidencias ordenadas por score.

In [ ]:
# Buscar multiples coincidencias para un termino ambiguo
matches = resolver.search("diabetes", max_results=5)

print(f"Resultados para 'diabetes' ({len(matches)} encontrados):\n")
for m in matches:
    confident = "OK" if m.is_confident else "REVISAR"
    print(f"  [{confident}] {m.concept.display}")
    system = m.concept.system.value
    print(f"        Codigo: {m.concept.code} | Sistema: {system} | Score: {m.score:.1f}")
    print()

## 3. Query Builder

El `FHIRQueryBuilder` permite construir queries FHIR complejas con una API fluent, sin manipular strings manualmente.

In [ ]:
from saludai_core.query_builder import (
    FHIRQueryBuilder,
    SortOrder,
    SummaryMode,
    snomed,
)

# Construir una query compleja:
# "Condiciones de hipertension, incluyendo datos del paciente, ordenadas por fecha"
query = (
    FHIRQueryBuilder("Condition")
    .where("code", snomed("59621000"))           # Hipertension esencial
    .where_string("clinical-status", "active")   # Solo activas
    .include("subject")                          # Incluir Patient referenciado
    .count(50)                                   # Maximo 50 resultados
    .sort("onset-date", SortOrder.DESC)          # Mas recientes primero
    .build()
)

params = query.to_params()
print(f"Resource type: {query.resource_type}")
print(f"Parametros generados:")
for key, value in params.items():
    print(f"  {key} = {value}")

### Ejecutar la query contra FHIR

Combinamos el QueryBuilder con el FHIRClient para ejecutar la consulta.

In [ ]:
async with FHIRClient(config=config) as client:
    bundle = await client.search(query.resource_type, params)

    conditions = [
        e["resource"] for e in bundle.get("entry", [])
        if e["resource"]["resourceType"] == "Condition"
    ]
    patients = {
        f"Patient/{e['resource']['id']}": e["resource"]
        for e in bundle.get("entry", [])
        if e["resource"]["resourceType"] == "Patient"
    }

    print(f"Pacientes con hipertension activa: {len(conditions)}\n")

    for cond in conditions[:5]:
        ref = cond.get("subject", {}).get("reference", "")
        patient = patients.get(ref, {})
        name = patient.get("name", [{}])[0]
        patient_name = " ".join(name.get("given", [])) + " " + name.get("family", "")
        code_display = cond.get("code", {}).get("coding", [{}])[0].get("display", "N/A")
        print(f"  {patient_name.strip()} — {code_display}")

### Contar recursos con _summary=count

Para obtener solo el total sin descargar datos, usamos `SummaryMode.COUNT`.

In [ ]:
async with FHIRClient(config=config) as client:
    # Contar pacientes, condiciones, observaciones y medicaciones
    resource_types = ["Patient", "Condition", "Observation", "MedicationRequest", "Encounter"]

    print("Recursos en el servidor FHIR:\n")
    for rt in resource_types:
        count_query = FHIRQueryBuilder(rt).summary(SummaryMode.COUNT).build()
        bundle = await client.search(rt, count_query.to_params())
        total = bundle.get("total", "?")
        print(f"  {rt}: {total}")

## 4. Locale Packs

Los locale packs permiten adaptar SaludAI a distintos paises. Argentina (`ar`) viene incluido por defecto.

In [ ]:
from saludai_core.locales import available_locales, load_locale_pack

print(f"Locales disponibles: {available_locales()}\n")

locale = load_locale_pack("ar")
print(f"Locale: Argentina")
print(f"  Sistemas de terminologia: {len(locale.terminology_systems)}")
print(f"  Perfiles FHIR: {len(locale.fhir_profiles)}")
print(f"  Sistemas de identificacion: {len(locale.identifier_systems)}")
print(f"  Extensiones: {len(locale.extensions)}")

print(f"\nPerfiles FHIR argentinos:")
for profile in locale.fhir_profiles[:5]:
    print(f"  - {profile.name}: {profile.description}")

print(f"\nSistemas de identificacion:")
for ids in locale.identifier_systems:
    print(f"  - {ids.name}: {ids.description}")

---

**Siguiente:** En el [notebook 02](02-agent-queries.ipynb) vemos como el agente combina estos componentes para responder preguntas clinicas en lenguaje natural.